In [2]:
import pandas as pd
import yaml
import re

# -------------------------------
# DATEIEN
# -------------------------------
excel_file = r"C:\Users\sickerti\Projekte\Inhaltsverzeichnisse\Ki_juni_2026_2.xlsx"
yaml_file = r"C:\Users\sickerti\Projekte\Inhaltsverzeichnisse\esk-idn.yaml"
output_dat = r"C:\Users\sickerti\Projekte\Inhaltsverzeichnisse\update_045Pjuni2.dat"

# -------------------------------
# LADEN
# -------------------------------
df = pd.read_excel(excel_file)

with open(yaml_file, "r", encoding="utf-8") as f:
    esk_map = yaml.safe_load(f)

# -------------------------------
# HELPER: ESK extrahieren
# -------------------------------
def extract_esk_codes(text):
    if not isinstance(text, str):
        return []
    
    match = re.search(r"ESK:\s*(.*?)\s*Begründung:", text, re.DOTALL)
    if not match:
        return []
    
    esk_part = match.group(1)

    codes = re.findall(r"ESK\.\d+(?:\.\d+|\.[A-Za-z]+)?", esk_part)
    
    return list(set(codes))  # doppelte vermeiden

def clean_relevance(text):
    if not isinstance(text, str):
        return ""
    
    # Teile an "Begründung:"
    text = text.replace("Relevanz:", "").strip()
    
    parts = text.split("Begründung:")
    
    relevance = parts[0].strip()
    reason = parts[1].strip() if len(parts) > 1 else ""
    
    if reason:
        return f"{relevance} - {reason}"
    return relevance
# -------------------------------
# DAT DATEI SCHREIBEN
# -------------------------------
with open(output_dat, "w", encoding="utf-8") as out:

    for _, row in df.iterrows():
        if str(row.get("hat_045P")).lower() != "false":
            continue

        idn = str(row.get("IDN")).strip()
        esk_text = row.get("ESK-Kategorie", "")

        if not idn:
            continue

        esk_codes = extract_esk_codes(esk_text)

        # -------------------------------
        # BLOCK START
        # -------------------------------
        out.write(f"{idn}\n")

        # FALL 1: ESK vorhanden
        if esk_codes:
            for esk in esk_codes:
                mapped_idn = esk_map.get(esk)

                if mapped_idn:
                    out.write(f"5320\t{mapped_idn}\n")
                else:
                    print(f"⚠️ Kein Mapping für {esk}")

        # FALL 2: KEINE ESK → nichts mit 5320

        relevance_text = clean_relevance(row.get("Relevanzbewertung", ""))

        out.write(f"4700\t|BSM|*{relevance_text}$hNVF-2606\n\t\n")

print(f"✅ Fertig: {output_dat}")

✅ Fertig: C:\Users\sickerti\Projekte\InhaltsverzeichnisseKI\update_045Pjuni2.dat
